# 06. Supabase pgvector 색인 구축

Steam 리뷰 원본에서 긍/부정 각 40만 건(총 80만 건)을 균형 샘플링해 정제한 뒤, Supabase pgvector(`reviews` 컬렉션)에 배치 단위로 적재하고 HNSW 인덱스를 생성한다.

**중요**: 임베딩 계산 + 80만 건 삽입은 로컬 환경에서 수 시간 걸릴 수 있다. 셀 4는 `output/06_pgvector_checkpoint.txt` 체크포인트를 기록하므로, 중단 후 재실행해도 마지막 지점부터 이어서 진행된다.

## 1. 환경 로드

`.env`에서 Supabase 연결 정보(`DATABASE_URL_DIRECT`)를 로드하고, 정제/색인 함수를 가져온다.

In [2]:
import os, sys, time
from pathlib import Path
import pandas as pd
from datasets import load_dataset
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

load_dotenv()
from src.config import DATABASE_URL_DIRECT, RANDOM_SEED
from src.data.pipeline import clean_reviews
from src.rag.index import reset_index, add_batch, get_game_counts

N_PER_LABEL = 400_000
BATCH_SIZE = 200  # 5000은 단일 INSERT문이 너무 커져 statement timeout 발생 (실측)
CHECKPOINT_PATH = str(ROOT / "output" / "06_pgvector_checkpoint.txt")

/Users/gomuseo/Desktop/Python/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. 원본 로드 + 균형 샘플링

HF `ksang/steamreviews`에서 긍정/부정 각 40만 건씩 샘플링해 섞는다.

In [3]:
ds = load_dataset("ksang/steamreviews", split="train")
df = ds.to_pandas()

pos = df[df["review_score"] == 1].sample(n=N_PER_LABEL, random_state=RANDOM_SEED)
neg = df[df["review_score"] == -1].sample(n=N_PER_LABEL, random_state=RANDOM_SEED)
sample = pd.concat([pos, neg]).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
print(len(sample))  # 800000 이어야 함

Repo card metadata block was not found. Setting CardData to empty.


800000


## 3. 정제

`clean_reviews`로 텍스트 정제 + 라벨 변환. `app_name` 컬럼을 유지해 게임별 메타데이터로 사용한다.

In [4]:
cleaned = clean_reviews(sample, "review_text", "review_score",
                        normalize=True, min_words=3, extra_cols=["app_name"])
cleaned["app_name"] = cleaned["app_name"].fillna("(unknown)")
print(len(cleaned))

626753


## 4. 체크포인트 기반 배치 적재

중단 시 재실행하면 `CHECKPOINT_PATH`에 기록된 지점부터 이어서 진행한다. 최초 실행 시에만 `reset_index`로 컬렉션을 초기화한다.

In [9]:
texts = cleaned["text"].tolist()
metadatas = [{"app_name": row.app_name, "label": int(row.label)}
             for row in cleaned.itertuples()]

start_batch = 0
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        start_batch = int(f.read().strip())
else:
    reset_index(DATABASE_URL_DIRECT)  # 최초 1회만 컬렉션 초기화

for s in range(start_batch, len(texts), BATCH_SIZE):
    e = min(s + BATCH_SIZE, len(texts))
    t0 = time.time()
    add_batch(texts[s:e], metadatas[s:e], DATABASE_URL_DIRECT)
    with open(CHECKPOINT_PATH, "w") as f:
        f.write(str(e))
    print(f"{e}/{len(texts)} ({time.time()-t0:.1f}s)")

95200/626753 (34.3s)
95400/626753 (23.7s)
95600/626753 (9.9s)
95800/626753 (20.5s)
96000/626753 (16.8s)
96200/626753 (10.8s)
96400/626753 (24.5s)
96600/626753 (21.8s)
96800/626753 (7.5s)
97000/626753 (6.9s)
97200/626753 (6.2s)
97400/626753 (56.2s)
97600/626753 (7.6s)
97800/626753 (49.2s)
98000/626753 (14.4s)
98200/626753 (39.4s)
98400/626753 (27.8s)
98600/626753 (15.7s)
98800/626753 (20.5s)
99000/626753 (8.7s)
99200/626753 (30.0s)
99400/626753 (49.8s)
99600/626753 (18.5s)
99800/626753 (11.7s)
100000/626753 (36.2s)
100200/626753 (24.4s)
100400/626753 (33.4s)
100600/626753 (30.9s)
100800/626753 (31.4s)
101000/626753 (36.0s)
101200/626753 (54.5s)
101400/626753 (89.5s)
101600/626753 (48.2s)
101800/626753 (72.6s)


Exception: Failed to create vector extension: (psycopg.OperationalError) connection failed: connection to server at "54.179.210.0", port 5432 failed: FATAL:  (EMAXCONNSESSION) max clients reached in session mode - max clients are limited to pool_size: 15
Multiple connection attempts failed. All failures were:
- host: 'aws-1-ap-southeast-1.pooler.supabase.com', port: 5432, hostaddr: '3.1.167.181': connection failed: connection to server at "3.1.167.181", port 5432 failed: FATAL:  (EMAXCONNSESSION) max clients reached in session mode - max clients are limited to pool_size: 15
connection to server at "3.1.167.181", port 5432 failed: FATAL:  (EMAXCONNSESSION) max clients reached in session mode - max clients are limited to pool_size: 15
- host: 'aws-1-ap-southeast-1.pooler.supabase.com', port: 5432, hostaddr: '13.213.241.248': connection failed: connection to server at "13.213.241.248", port 5432 failed: FATAL:  (EMAXCONNSESSION) max clients reached in session mode - max clients are limite

## 5. HNSW 인덱스 생성

적재가 모두 끝난 뒤 1회만 실행한다.

In [5]:
import psycopg

with psycopg.connect(DATABASE_URL_DIRECT) as conn:
    conn.autocommit = True
    with conn.cursor() as cur:
        cur.execute("""
            CREATE INDEX IF NOT EXISTS langchain_pg_embedding_hnsw_idx
            ON langchain_pg_embedding
            USING hnsw (embedding vector_cosine_ops);
        """)
        cur.execute("ANALYZE langchain_pg_embedding;")
print("index created")

index created


## 6. 검증 — 게임별 집계

최소 20건 이상 리뷰가 있는 게임 수와 상위 10개를 확인한다.

In [6]:
counts = get_game_counts(min_count=20)
print(len(counts), counts[:10])

903 [('PAYDAY 2', 2143), ('Dota 2', 1180), ('Terraria', 1088), ('Grand Theft Auto V', 934), ('Left 4 Dead 2', 770), ('Warframe', 747), ('Rocket League', 729), ('Undertale', 715), ('Call of Duty: Black Ops III', 711), ("No Man's Sky", 646)]
